# export HSA_OVERRIDE_GFX_VERSION=10.3.0
# ollama serve


In [ ]:
import ollama

response = ollama.chat(model='deepseek-r1:7b', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])

In [ ]:
from ollama import chat

messages = [
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
]

for chunk in chat('gemma3:12b-it-qat', messages=messages, stream=True):
  print(chunk['message']['content'], end='', flush=True)

In [1]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_db.db")

In [2]:
if client.has_collection(collection_name="hybrid_collection"):
    print("collection exists")

collection exists


In [ ]:
import chainlit as cl
from pymilvus import MilvusClient
import requests
from vector_db import query_hybrid_search
from ollama import chat

message_history = []

@cl.on_message
async def handle_message(message: cl.Message):
    query = message.content

    message_history.append({"role": "user", "content": query})

    docs = query_hybrid_search(query)
    context = "\n".join(docs)

    message_history.append({"role": "system", "content": context})

    response = chat(
        model='gemma3:4b-it-qat',
        messages=message_history
    )

    assistant_reply = response['message']['content']

    message_history.append({"role": "assistant", "content": assistant_reply})

    await cl.Message(content=assistant_reply).send()


In [ ]:
import importlib
import vector_db
importlib.reload(vector_db)
from vector_db import query_hybrid_search

/home/grzegorz/python/pharmai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from vector_db import query_hybrid_search, extract_ids, load_texts_from_ids
query = """jakie są wskazania do stosowania leku w postaci kapsułki żelatynowej?"""

db_result = query_hybrid_search(query, limit=1)
db_result = extract_ids(db_result)
db_result = load_texts_from_ids(db_result)
db_result.values()

/home/grzegorz/python/pharmai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_values(["## CHARAKTERYSTYKA PRODUKTU LECZNICZEGO\n\n## 1. NAZWA PRODUKTU LECZNICZEGO\n\nObsidan, 100 mg, kapsułki dojelitowe, twarde\n\n## 2. SKŁAD JAKOŚCIOWY I ILOŚCIOWY\n\n1 kapsułka zawiera:\n\n567,7 mg kompleksu żelaza siarczanu (II) pięciowodnego z glicyną , co odpowiada 100 mg jonów żelaza ( Fe 2+ ).\n\nPełny wykaz substancji pomocniczych, patrz punkt 6.1.\n\n## 3. POSTAĆ FARMACEUTYCZNA\n\nKapsułka dojelitowa, twarda\n\nK apsułka żelatynowa , twarda, rozmiar 0 (około 21,7 mm x 7,5 mm) z czekoladowobrązowym, nieprzezroczystym wieczkiem bez nadruku i pomarańczowym, nieprzezroczystym korpusem bez nadruku\n\nZawartość kapsułki: szaro brązowawe - granulki powlekane, dojelitowe\n\n## 4. SZCZEGÓŁOWE DANE KLINICZNE\n\n## 4.1 Wskazania do stosowania\n\nNiedobór żelaza.\n\nProdukt leczniczy Obsidan jest wskazany do stosowania u osób dorosłych i dzieci w wieku od 6 lat (o minimalnej masie ciała 20 kg).\n\n## 4.2 Dawkowanie i sposób podawania\n\n## Dawkowanie\n\nN ależy dostosować dawko

In [26]:
headers = """NAZWA PRODUKTU LECZNICZEGO, SKŁAD JAKOŚCIOWY I ILOŚCIOWY, POSTAĆ FARMACEUTYCZNA, SZCZEGÓŁOWE DANE KLINICZNE, WŁAŚCIWOŚCI FARMAKOLOGICZNE, DANE FARMACEUTYCZNE, PODMIOT ODPOWIEDZIALNY POSIADAJĄCY POZWOLENIE NA DOPUSZCZENIE DO OBROTU, NUMER POZWOLENIA NA DOPUSZCZENIE DO OBROTU, DATA WYDANIA PIERWSZEGO POZWOLENIA NA DOPUSZCZENIE DO OBROTU I DATA PRZEDŁUŻENIA POZWOLENIA, DATA ZATWIERDZENIA LUB CZĘŚCIOWEJ ZMIANY TEKSTU CHARAKTERYSTYKI PRODUKTU LECZNICZEGO"""

In [ ]:
from ollama import chat

q = """jakie są wskazania do stosowania leku w postaci kapsułki żelatynowej? czy zapobiega odrzuceniu przeszczepu?"""

messages = [
  {
    'role': 'user',
    'content': f"To answer the question: {q} what part of the medicine characteristic would be from the following: {headers}? Answer in polish giving only the the part titles that you need separated by commas.",
  },
]
for chunk in chat('gemma3:4b-it-qat', messages=messages, stream=True):
    print(chunk['message']['content'], end='', flush=True)

In [14]:
collection_info = client.describe_collection("hybrid_collection")

fields = collection_info["fields"]
for field in fields:
    print(field["name"])

id
text
dense_vector
sparse_vector
availability_categories


In [ ]:
results = client.query(
    collection_name="hybrid_collection",
    filter="",
    output_fields=["id"],
    limit=1000,
)

for item in results:
    print(item["id"])

100002817
100021312
100047665
100078772
100082561
100083610
100141393
100205348
100231966
100296428
100302459
100332549
100351972
100358543
100388030
100396696
100409795
100420880
100433871
100445199


In [7]:
query = """jakie są okrągłe tabletki na dławice piersiową? czy ten lek można stosować w ciąży?"""

db_result = query_hybrid_search(query)[0][0]['entity']['text']
db_result

'## CHARAKTERYSTYKA PRODUKTU LECZNICZEGO\n\n## 1. NAZWA PRODUKTU LECZNICZEGO\n\nNaxalgan, 75 mg, kapsułki, twarde\n\nNaxalgan, 150 mg, kapsułki, twarde\n\nNaxalgan, 300 mg, kapsułki, twarde\n\n## 2. SKŁAD JAKOŚCIOWY I ILOŚCIOWY\n\nNaxalgan, 75 mg, kapsułki, twarde\n\nKażda kapsułka twarda zawiera 75 mg pregabaliny ( Pregabalinum ).\n\nNaxalgan, 150 mg, kapsułki, twarde\n\nKażda kapsułka twarda zawiera 150 mg pregabaliny ( Pregabalinum ).\n\nNaxalgan, 300 mg, kapsułki, twarde Każda kapsułka twarda zawiera 300 mg pregabaliny ( Pregabalinum ).\n\nSubstancje pomocnicze o znanym działaniu\n\nNaxalgan, 75 mg, kapsułki, twarde Każda kapsułka twarda zawiera także 8,25 mg laktozy jednowodnej.\n\nNaxalgan, 150 mg, kapsułki, twarde\n\nKażda kapsułka twarda zawiera także 16,50 mg laktozy jednowodnej.\n\nNaxalgan, 300 mg, kapsułki, twarde\n\nKażda kapsułka twarda zawiera także 33 mg laktozy jednowodnej.\n\nPełny wykaz substancji pomocniczych, patrz punkt 6.1.\n\n## 3. POSTAĆ FARMACEUTYCZNA\n\nKapsu

In [59]:
query = """jakie są okrągłe tabletki na dławice piersiową? czy ten lek można stosować w ciąży?"""

db_result = query_hybrid_search(query)
db_result = extract_ids(db_result)
db_result = load_texts_from_ids(db_result)
db_result = db_result[100063871]
db_result
# split in half
# db_result = db_result.split('\n')
# db_result = '\n'.join(db_result[:len(db_result)//20])
# db_result = db_result*10
# db_result

"## CHARAKTERYSTYKA PRODUKTU LECZNICZEGO\n\n## 1. NAZWA PRODUKTU LECZNICZEGO\n\nStediril 30, 150 mikrogramów (µg) + 30 mikrogramów (µg), tabletki drażowane\n\n## 2. SKŁAD JAKOŚCIOWY I ILOŚCIOWY\n\nKażda tabletka drażowana zawiera 30 mikrogramów (µg) etynyloestradiolu ( Ethinylestradiolum ) i 150 mikrogramów (µg) lewonorgestrelu ( Levonorgestrelum ).\n\nSubstancje pomocnicze o znanym działaniu:\n\nKażda tabletka drażowana zawiera 32,970 mg laktozy jednowodnej i 22,023 mg sacharozy.\n\nPełny wykaz substancji pomocniczych, patrz punkt 6.1.\n\n## 3. POSTAĆ FARMACEUTYCZNA\n\nTabletki drażowane.\n\nStediril 30 to biała, błyszcząca, powlekana otoczką cukrową tabletka o gładkiej powierzchni, która jest oznaczona wg 21-dniowego porządku.\n\n## 4. SZCZEGÓŁOWE DANE KLINICZNE\n\n## wania\n\nZapobieganie ciąży.\n\n## 4.2 Dawkowanie i sposób podawania\n\n## Jak stosować produkt Stediril 30\n\nTabletki 1-21 zawierają substancje czynne (tabletki aktywne).\n\nTabletki należy przyjmować, zgodnie z kieru

In [ ]:
from ollama import chat

messages = [
    {
        "role": "user",
        "content": f"Na podstawie poniższych informacji:\n\n{db_result}\n\nOdpowiedz na pytanie: {query}. Odpowiadasz osobie, która nie zna treści dokumentu",
    }
]

for chunk in chat('gemma3:4b-it-qat', messages=messages, stream=True):
    print(chunk['message']['content'], end='', flush=True)

In [ ]:
# wyswietlenie id leku z db_result
query_hybrid_search(query)[0][0]['id']


100409795

In [25]:
db_result[0]['entity']['text']

"## CHARAKTERYSTYKA PRODUKTU LECZNICZEGO\n\n## 1. NAZWA PRODUKTU LECZNICZEGO\n\nDailiport, 0,5 mg, kapsułki o przedłużonym uwalnianiu , twarde\n\nDailiport, 1 mg, kapsułki o przedłużonym uwalnianiu , twarde\n\nDailiport, 2 mg, kapsułki o przedłużonym uwalnianiu , twarde\n\nDailiport, 3 mg, kapsułki o przedłużonym uwalnianiu , twarde\n\nDailiport, 5 mg, kapsułki o przedłużonym uwalnianiu , twarde\n\n## 2. SKŁAD JAKOŚCIOWY I ILOŚCIOWY\n\n## Dailiport, 0,5 mg\n\nKażda kapsułka o przedłużonym uwalnianiu , twarda zawiera 0,5 mg takrolimusu ( Tacrolimusum ) w postaci takrolimusu jednowodnego .\n\n## Substancj e pomocnicz e o znanym działaniu\n\nKażda kapsułka o przedłużonym uwalnianiu, twarda zawiera 51 mg laktozy (w postaci laktozy jednowodnej ).\n\nKażda kapsułka o przedłużonym uwalnianiu, twarda zawiera 5,4 mikrograma żółcieni pomarańczowej FCF (E 110).\n\nKażda kapsułka o przedłużonym uwalnianiu, twarda zawiera 0,4 mikrograma czerwieni A llura AC (E 129).\n\nKażda kapsułka o przedłużonym

In [ ]:
import importlib
import vector_db
importlib.reload(vector_db)
from vector_db import query_hybrid_search

/home/grzegorz/python/pharmai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
from vector_db import query_hybrid_search, extract_ids, load_texts_from_ids
query = """jakie są wskazania do stosowania leku w postaci kapsułki żelatynowej?"""

db_result = query_hybrid_search(query)
db_result = extract_ids(db_result)
db_result = load_texts_from_ids(db_result)
db_result

first_id = next(iter(db_result))
print(f"Key: {first_id}")
first_value = db_result[first_id]
print(f"Value: {first_value}")

/home/grzegorz/python/pharmai/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Key: 100366347
Value: ## CHARAKTERYSTYKA PRODUKTU LECZNICZEGO

## 1. NAZWA PRODUKTU LECZNICZEGO

Calfos, 0,266 mg, kapsułki miękkie

## 2. SKŁAD JAKOŚCIOWY I ILOŚCIOWY

Każda kapsułka miękka zawiera 0,266 mg kalcyfediolu jednowodnego, co odpowiada 0,255 mg kalcyfediolu.

Substancje pomocnicze o znanym działaniu:

Każda kapsułka miękka zawiera 5 mg etanolu, 22 mg sorbitolu (E 420) i 1 mg żółcieni pomarańczowej  (E 110).

Pełny wykaz substancji pomocniczych, patrz punkt 6.1.

## 3. POSTAĆ FARMACEUTYCZNA

## Kapsułka miękka

Pomarańczowe, owalne, żelatynowe, 15 mm na 9 mm kapsułki miękkie zawierające przejrzysty płyn o niskiej lepkości,  wolny od cząstek stałych.

## 4. SZCZEGÓŁOWE DANE KLINICZNE

## 4.1 Wskazania do stosowania

Leczenie niedoboru witaminy D (tj. stężenie 25(OH)D &lt; 25 nmol/l) u osób dorosłych.

Zapobieganie niedoborowi witaminy D u osób dorosłych ze zidentyfikowanym ryzykiem, takim jak zespół złego wchłaniania, zespół zaburzeń mineralnych i kostnych związanych z przewl